# eManual Experiment Runner — OpenRouter variant (parallel to the Groq run)

Same emanual RAG sweep as `emanual_experiment_notebook.ipynb`, but all LLM calls
go through **OpenRouter** instead of Groq. Run this notebook alongside the Groq
one to parallelize without sharing Groq's rate limit.

Driven by `experiment_configs/emanual_openrouter_experiment.yaml` and the configs
under `rag-experiments/emanual-openrouter-experiment/config/`. It uses its own
`temp/`, `reports/`, and `cache_openrouter/` dirs so the two runs never collide.

**Requires** `OPENROUTER_API_KEY` (comma-separate multiple keys to rotate). The
`openai` package must be installed (it is added to the install cell below).


## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas -q')



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [2]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
print('core.registry   loaded from:', _reg.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))


/Users/bhupendra.bhoi/pandas_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current directory: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support
experiment_runner loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/experiment/experiment_runner.py
core.registry   loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/core/registry.py
HuggingFace token loaded: True
Groq API key loaded: True
OpenRouter API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [3]:
EXPERIMENT_CONFIG_PATH = project_root / "experiment_configs/emanual_experiment.yaml"

experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Temp Dir:    {experiment_config.temp_dir}")
print(f"  Cache:       {experiment_config.cache}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  rag-experiments/emanual-experiment/config
  Report Dir:  rag-experiments/emanual-experiment/reports
  Temp Dir:    rag-experiments/emanual-experiment/temp
  Cache:       {'enabled': True, 'cache_dir': './cache'}
  Num Queries: 20
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'emanual', 'split': 'test', 'limit': 132}

Data Parser:
  Type: noop


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [4]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


In [5]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/emanual (test)...
Loaded 132 samples

✅ Data loaded successfully!
  Documents: 101 parsed documents
  Raw Data:  132 samples

First Sample:
  Question: I want to  enter into Ambient mode. How can I do that?...
  Documents: 3


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [6]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f'Loaded {len(configs)} RAG pipeline configurations:')
for cfg in configs:
    searches = ' + '.join(s.type.value for s in cfg.retrieval.search.searches)
    fusion = cfg.retrieval.fusion.type.value if cfg.retrieval.fusion else '-'
    rerank = cfg.retrieval.rerank.type.value if cfg.retrieval.rerank else '-'
    qx = cfg.retrieval.query_transform.type.value if cfg.retrieval.query_transform else '-'
    gc = cfg.generation.config
    model = gc.get('model') if isinstance(gc, dict) else getattr(gc, 'model', None)
    print(f'  - {cfg.name}')
    print(f'      chunking={cfg.chunking.type.value}  embed={cfg.embedding.type.value}')
    print(f'      search=[{searches}]  fusion={fusion}  rerank={rerank}  q_transform={qx}')
    print(f'      generation_model={model}')


Loaded 9 RAG pipeline configurations:
  - emanual_v11_role_aware_bgem3
      chunking=sentence  embed=sentence_transformer
      search=[dense + sparse]  fusion=rrf  rerank=cross_encoder  q_transform=-
      generation_model=llama-3.3-70b-versatile
  - emanual_v1_baseline
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=llama-3.3-70b-versatile
  - emanual_v2_hybrid_wsum
      chunking=fixed_word  embed=sentence_transformer
      search=[dense + sparse]  fusion=weighted_sum  rerank=-  q_transform=-
      generation_model=llama-3.3-70b-versatile
  - emanual_v3_embed_bge
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=llama-3.3-70b-versatile
  - emanual_v4_chunk_sentence
      chunking=sentence  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=llama-3.3-70b-versatile

## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [7]:
get_ipython().system('pip install rank_bm25 -q')

# Run experiments


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 9 configurations from 0 to 20 queries...
Parallel mode: False
Loading embedding model: BAAI/bge-m3


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 60778.00it/s]


Loading reranker model: BAAI/bge-reranker-v2-m3


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6343.24it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 5sUsing key #0: ****Yzkz
Using key #0: ****Yzkz
Progress: 2/20 (10.0%) | QPS: 0.17 | ETA: 1.8m | Elapsed: 12ssUsing key #0: ****Yzkz
Using key #0: ****Yzkz
Progress: 3/20 (15.0%) | QPS: 0.19 | ETA: 1.5m | Elapsed: 16sUsing key #0: ****Yzkz
Progress: 4/20 (20.0%) | QPS: 0.12 | ETA: 2.3m | Elapsed: 34sUsing key #0: ****Yzkz
Progress: 5/20 (25.0%) | QPS: 0.09 | ETA: 2.7m | Elapsed: 53sUsing key #0: ****Yzkz
Progress: 6/20 (30.0%) | QPS: 0.08 | ETA: 2.8m | Elapsed: 1.2mUsing key #0: ****Yzkz
Progress: 6/20 (30.0%) | QPS: 0.08 | ETA: 2.8m | Elapsed: 1.2m================================================================================
429 on ****Yzkz
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kq21xc5dec58dfyhejka65td` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11290, Requested 4429. Please try again in 18.595s. Need more tokens? Up

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8090.59it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #3: ****isl2
Using key #3: ****isl2
Using key #3: ****isl2
Using key #3: ****isl2
Using key #3: ****isl2
Using key #3: ****isl2
Using key #3: ****isl2
Progress: 5/20 (25.0%) | QPS: 4.98 | ETA: 3s | Elapsed: 1sUsing key #3: ****isl2
Using key #3: ****isl2
Progress: 7/20 (35.0%) | QPS: 3.48 | ETA: 4s | Elapsed: 2sUsing key #3: ****isl2
Using key #3: ****isl2
Progress: 9/20 (45.0%) | QPS: 1.79 | ETA: 6s | Elapsed: 5sUsing key #3: ****isl2
Progress: 10/20 (50.0%) | QPS: 0.99 | ETA: 10s | Elapsed: 10sUsing key #3: ****isl2
429 on ****isl2
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kyayrpkneawrsb0tg3sgy3r7` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11651, Requested 1019. Please try again in 3.35s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6322.70it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #0: ****Yzkz
Using key #0: ****Yzkz
Using key #0: ****Yzkz
Using key #0: ****Yzkz
Using key #0: ****Yzkz
Using key #0: ****Yzkz
Progress: 4/20 (20.0%) | QPS: 3.99 | ETA: 4s | Elapsed: 1sUsing key #0: ****Yzkz
Using key #0: ****Yzkz
Using key #0: ****Yzkz
Progress: 8/20 (40.0%) | QPS: 3.98 | ETA: 3s | Elapsed: 2sUsing key #0: ****Yzkz
Progress: 8/20 (40.0%) | QPS: 1.99 | ETA: 6s | Elapsed: 4sUsing key #0: ****Yzkz
Progress: 9/20 (45.0%) | QPS: 1.12 | ETA: 10s | Elapsed: 8s================================================================================
429 on ****Yzkz
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kq21xc5dec58dfyhejka65td` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11644, Requested 1051. Please try again in 3.475s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing

## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [9]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

Found 9 configs with JSONL files
Using key #2: ****yNwZ
Using key #2: ****yNwZ
Using key #2: ****yNwZ
Using key #2: ****yNwZ
429 on ****yNwZ
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kyayk6txfny8n7ahf9xc9x96` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11793, Requested 5902. Please try again in 28.475s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sun, 26 Jul 2026 05:00:14 GMT
  content-type: application/json
  content-length: 385
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 29
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 962
  x-ratelimit-remaining-tokens: 207
  x-ratelimit-reset

Record 6 evaluation failed: All Groq API keys are currently rate limited.
Record 12 evaluation failed: All Groq API keys are currently rate limited.
Record 13 evaluation failed: All Groq API keys are currently rate limited.
Record 14 evaluation failed: All Groq API keys are currently rate limited.
Record 15 evaluation failed: All Groq API keys are currently rate limited.
Record 16 evaluation failed: All Groq API keys are currently rate limited.
Record 17 evaluation failed: All Groq API keys are currently rate limited.
Record 18 evaluation failed: All Groq API keys are currently rate limited.
Record 19 evaluation failed: All Groq API keys are currently rate limited.
Record 20 evaluation failed: All Groq API keys are currently rate limited.


429 on ****7gBP
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kyaz4w9sf2n85wmxftrm7c8d` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10494, Requested 1758. Please try again in 1.26s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sun, 26 Jul 2026 05:08:26 GMT
  content-type: application/json
  content-length: 383
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 2
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 952
  x-ratelimit-remaining-tokens: 1506
  x-ratelimit-reset-requests: 1h9m7.2s
  x-ratelimit-reset-tokens: 52.47s
  x-request-id: req_01kyed64hre5mb9cx7sez3y4pr
  via: 1.1 google
  cf-ca

Record 4 evaluation failed: All Groq API keys are currently rate limited.
Record 5 evaluation failed: All Groq API keys are currently rate limited.
Record 6 evaluation failed: All Groq API keys are currently rate limited.
Record 7 evaluation failed: All Groq API keys are currently rate limited.
Record 8 evaluation failed: All Groq API keys are currently rate limited.
Record 9 evaluation failed: All Groq API keys are currently rate limited.
Record 10 evaluation failed: All Groq API keys are currently rate limited.
Record 11 evaluation failed: All Groq API keys are currently rate limited.
Record 12 evaluation failed: All Groq API keys are currently rate limited.
Record 13 evaluation failed: All Groq API keys are currently rate limited.
Record 14 evaluation failed: All Groq API keys are currently rate limited.
Record 15 evaluation failed: All Groq API keys are currently rate limited.
Record 16 evaluation failed: All Groq API keys are currently rate limited.
Record 17 evaluation failed: Al

Using key #5: ****7gBP
429 on ****7gBP
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kyaz4w9sf2n85wmxftrm7c8d` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99079, Requested 2135. Please try again in 17m28.896s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sun, 26 Jul 2026 05:08:42 GMT
  content-type: application/json
  content-length: 386
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 1049
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 949
  x-ratelimit-remaining-tokens: 506
  x-ratelimit-reset-requests: 1h13m26.4s
  x-ratelimit-reset-tokens: 57.47s
  x-request-id: req_01kyed6mrgef0tm082h92x

Record 3 evaluation failed: All Groq API keys are currently rate limited.
Record 1 evaluation failed: All Groq API keys are currently rate limited.
Record 2 evaluation failed: All Groq API keys are currently rate limited.
Record 3 evaluation failed: All Groq API keys are currently rate limited.
Record 4 evaluation failed: All Groq API keys are currently rate limited.
Record 5 evaluation failed: All Groq API keys are currently rate limited.
Record 6 evaluation failed: All Groq API keys are currently rate limited.
Record 7 evaluation failed: All Groq API keys are currently rate limited.
Record 8 evaluation failed: All Groq API keys are currently rate limited.
Record 9 evaluation failed: All Groq API keys are currently rate limited.
Record 10 evaluation failed: All Groq API keys are currently rate limited.
Record 11 evaluation failed: All Groq API keys are currently rate limited.
Record 13 evaluation failed: All Groq API keys are currently rate limited.
Record 12 evaluation failed: All Gr

429 on ****7gBP
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kyaz4w9sf2n85wmxftrm7c8d` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99078, Requested 1843. Please try again in 13m15.743999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Sun, 26 Jul 2026 05:08:44 GMT
  content-type: application/json
  content-length: 392
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 796
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-requests: 1000
  x-ratelimit-limit-tokens: 12000
  x-ratelimit-remaining-requests: 949
  x-ratelimit-remaining-tokens: 750
  x-ratelimit-reset-requests: 1h13m26.4s
  x-ratelimit-reset-tokens: 56.25s
  x-request-id: req_01kyed6nype1c80144ctgctytn
  x-should-re

## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [10]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: rag-experiments/emanual-experiment/reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [11]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: emanual_v11_role_aware_bgem3

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `emanual_v11_role_aware_bgem3`

**name**: emanual_v11_role_aware_bgem3  •  **mode**: test  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'sentence', 'config': {'max_words': 380, 'overlap_sentences': 2}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'BAAI/bge-m3', 'dimension': 1024}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 1024}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 40}}, {'type': 'sparse', 'config': {'top_k': 40}}]}, 'query_transform': None, 'fusion': {'type': 'rrf', 'config': {'top_k': 40, 'k': 60}}, 'rerank': {'type': 'cross_encoder', 'config': {'model_name': 'BAAI/bge-reranker-v2-m3', 'top_k': 15}}}  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'llama-3.3-70b-versatile', 'temperature': 0.0, 'max_tokens': 512, 'system_prompt': 'You are a consumer-electronics product-support assistant. Answer questions\nabout device features and settings using ONLY the provided user-manual passages.\n\nCRITICAL RULES:\n1. Answer ONLY from the passages provided. Do not use outside knowledge.\n2. Give exact, step-by-step instructions. Preserve menu paths, button names,\n   setting names, and on-screen labels verbatim\n   (e.g. "Settings > Support > Self Diagnosis > Signal Information").\n3. Be concise and procedural - give the steps, not background.\n4. If the passages do not contain the answer, respond with exactly:\n   "The passages do not provide sufficient information to answer this question."\n5. Every step must be directly supported by the passages provided.\n', 'user_prompt': 'Passages:\n{context}\n\nQuestion: {query}\n\nAnswer (from passages only, give exact steps and menu paths):\n'}}  •  **evaluation**: {'type': 'trace', 'provider': 'groq', 'config': {'model': 'llama-3.3-70b-versatile', 'temperature': 0.0, 'max_tokens': 2000}}  •  **cache**: {'enabled': True, 'cache_dir': './cache'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,I want to enter into Ambient mode. How can I do that?,"[{'text': '', 'title': '', 'content': 'beautiful screens, various visual inf...","To enter Ambient Mode, press the button. \n\nNote: The passages do not speci...",None,0.0204,None,None,0.0408,None,None,1.0000,None,None,1.0,None
1,Where do I find signal information ?,"[{'text': '', 'title': '', 'content': 'Checking digital channel signal info ...",Settings > Support > Self Diagnosis > Signal Information > Try Now,None,0.1111,None,None,0.1111,None,None,1.0000,None,None,1.0,None
2,How can I view the channels that are serached by auto program function and H...,"[{'text': '', 'title': '', 'content': 'up and down directional buttons to mo...",To view the channels that are searched by the auto program function: \nPress...,None,0.4211,None,None,0.1579,None,None,0.3750,None,None,1.0,None
3,Can I configure Tint?,"[{'text': '', 'title': '', 'content': 'Configuring advanced picture settings...",Settings > Picture > Expert Settings > Try Now > Tint (G/R) > Try Now,None,0.0323,None,None,0.0323,None,None,1.0000,None,None,1.0,None
4,How do I fix the missing/wrong color issue ?,"[{'text': '', 'title': '', 'content': 'Solution Power Saving Mode Settings G...","To fix the missing/wrong color issue, follow these steps:\n\n1. Run the ""Sta...",None,0.0893,None,None,0.1786,None,None,0.4000,None,None,1.0,None
5,How do I fix blurring issues on TV screen?,"[{'text': '', 'title': '', 'content': 'on your TV are correct but just a lit...","To fix blurring issues on the TV screen, follow these steps: \nSettings > Pi...",None,0.0357,None,None,0.0357,None,None,1.0000,None,None,1.0,None
6,What is the use of universal guide?,"[{'text': '', 'title': '', 'content': 'Using the Universal Guide App. Search...",The Universal Guide is an app that allows you to search for and enjoy variou...,None,0.2778,None,None,0.1667,None,None,0.6000,None,None,1.0,None
7,What is the feature of Bixby guide?,"[{'text': '', 'title': '', 'content': 'Running Bixby. Press and hold the but...","To view the Bixby guide, \n1. Press the button on your Samsung Smart Remote ...",None,0.1333,None,None,0.1333,None,None,1.0000,None,None,1.0,None
8,How to launch the last used app automatically?,"[{'text': '', 'title': '', 'content': 'Launching the last used app automatic...","To launch the last used app automatically, follow these steps: \nSettings > ...",None,0.0682,None,None,0.0455,None,None,0.6667,None,None,1.0,None
9,Where do I find the list of my favorite channels?,"[{'text': '', 'title': '', 'content': 'Creating a Personal Favorites List. D...","To find the list of your favorite channels, follow these steps: \n\nPress th...",None,0.4167,None,None,0.3333,None,None,0.8000,None,None,1.0,None


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,NaN,0.1951,NaN,0.1870,NaN
1,utilization_score,NaN,0.1465,NaN,0.1258,NaN
2,completeness_score,NaN,0.7481,NaN,0.2503,NaN
3,adherence_score,NaN,0.9000,NaN,0.3000,NaN


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [12]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: rag-experiments/emanual-experiment/reports/comparison.csv


In [13]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,emanual_v8_hyde,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,emanual_v3_embed_bge,0.3013,0.1578,0.1467,0.1043,0.4333,0.3532,0.60,0.40
2,emanual_v6_rerank_only,0.2358,0.0970,0.1443,0.0448,0.5765,0.2377,0.30,0.70
3,emanual_v1_baseline,0.3785,0.2318,0.1678,0.1211,0.3797,0.4094,0.55,0.55
4,emanual_v11_role_aware_bgem3,0.0704,0.1369,0.0281,0.1184,0.4203,0.3594,0.35,0.65
5,emanual_v7_hybrid_rerank,0.2310,0.1652,0.0903,0.0799,0.4405,0.5595,1.00,0.00
6,emanual_v2_hybrid_wsum,0.1824,0.1257,0.0723,0.0937,0.3582,0.3899,0.40,0.70
7,emanual_v4_chunk_sentence,0.3326,0.2063,0.1429,0.1005,0.4195,0.4257,0.45,0.65
8,emanual_v5_hybrid_rrf,0.2052,0.1510,0.0677,0.0979,0.3232,0.4354,0.45,0.55


## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons